# Limpieza y Transformación (ETL) del año 2020
Usando el dataframe ya en limpio del año 2019 decidi contraponer el año 2020 y 2021 para luego al final, cuando cree visualizaciones, poder tener un contexto de pre pandemia, pandemia en si y post pandemia para medir.

In [1]:
import pandas as pd
import numpy as np

path_clean = "../data/clean/"
df_2019_clean = pd.read_csv(path_clean + "historico_2019_clean.csv", sep=";", encoding="utf-8-sig")
schema_2019 = df_2019_clean.columns.tolist()

print("Columnas schema 2019:", len(schema_2019))
schema_2019

Columnas schema 2019: 1


['periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,hora_desde,hora_hasta,dia_semana,mes,dia_mes,es_fin_semana']

# IMPORTANTE: DATA SCHEMA
Opte por usar el df limpio del 2019 como esquema inicial para respetar la estructura de datos presentada en aquel archivo. Tanto en este file como en el próximo a crear , año 2021, tendran que respetar las columnas de datos propuestas por el 2019 y en caso de tener inconsistencias, completar como lo amerite.


In [2]:
#Carga datos crudos

path_raw = "../data/raw/"
df_2020 = pd.read_csv(path_raw + "historico_2020.csv")

print(df_2020.shape)
df_2020.head(2)


(5781006, 10)


,FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,pax_pagos,pax_pases_pagos,pax_franq,pax_TOTAL
0,01/01/2020,08:00:00,08:15:00,LineaA,LineaA_Acoyte_N_Turn01,Acoyte,1.0,0.0,0.0,1.0
1,01/01/2020,08:00:00,08:15:00,LineaA,LineaA_Carabobo_E_Turn02,Carabobo,6.0,0.0,0.0,6.0


In [3]:
cols_2020 = set(df_2020.columns)
cols_2019 = set(schema_2019)

faltan_en_2020 = sorted(list(cols_2019 - cols_2020))
sobran_en_2020 = sorted(list(cols_2020 - cols_2019))

faltan_en_2020, sobran_en_2020


(['periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,hora_desde,hora_hasta,dia_semana,mes,dia_mes,es_fin_semana'],
 ['DESDE',
  'ESTACION',
  'FECHA',
  'HASTA',
  'LINEA',
  'MOLINETE',
  'pax_TOTAL',
  'pax_franq',
  'pax_pagos',
  'pax_pases_pagos'])

### Diferencias estructurales entre los datasets 2019 y 2020

Durante la comparación del esquema del dataset correspondiente al año 2020 con
el esquema de referencia definido a partir de 2019, noté diferencias
estructurales relevantes que justifican la implementación de un proceso ETL
específico para este período.

En particular, el dataset 2020 presenta:
- Nombres de columnas en mayúsculas (`FECHA`, `LINEA`, `ESTACION`, `MOLINETE`),
  a diferencia del formato estandarizado en 2019.
- Un cambio en la nomenclatura de la métrica agregada de pasajeros, donde
  `total` en 2019 aparece como `pax_TOTAL` en 2020.
- Ausencia de columnas derivadas presentes en el dataset limpio de 2019
  (por ejemplo: `periodo`, `hora_desde`, `hora_hasta`, `dia_semana`, `mes`,
  `dia_mes`, `es_fin_semana`), las cuales no forman parte del dataset crudo
  y deben ser generadas durante la etapa de transformación.

Estas diferencias no implican pérdida de información, sino variaciones en el
formato y estructura del dataset, probablemente asociadas a cambios en los
procesos de generación o publicación de los datos durante el período pandémico.

Por este motivo, adopto un enfoque de limpieza y transformación específico
para el año 2020, utilizando el esquema del dataset 2019 como referencia
estructural.
Este enfoque me permite alinear los datos a un formato común,
garantizando consistencia y comparabilidad temporal sin forzar información
inexistente.


In [11]:
df_2020["desde"].isna().sum(), df_2020["hasta"].isna().sum()

(np.int64(1836), np.int64(1836))

In [4]:
rename_map = {
    "FECHA": "fecha",
    "DESDE": "desde",
    "HASTA": "hasta",
    "LINEA": "linea",
    "MOLINETE": "molinete",
    "ESTACION": "estacion",
    "pax_TOTAL": "total",
}

df_2020 = df_2020.rename(columns=rename_map)
df_2020.columns


Index(['fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 'pax_pagos',
       'pax_pases_pagos', 'pax_franq', 'total'],
      dtype='object')

### Alineación del esquema 2020 con el dataset de referencia 2019

Luego de estandarizar los nombres de columnas del dataset 2020, hice una
comparación contra el esquema de referencia definido a partir del dataset limpio
de 2019.

El análisis muestra que:
- No existen columnas adicionales en el dataset 2020 que no estén presentes en
  el esquema de referencia.
- Las únicas columnas ausentes corresponden a variables derivadas
  (`periodo`, `mes`, `dia_mes`, `dia_semana`, `es_fin_semana`,
  `hora_desde`, `hora_hasta`), las cuales no forman parte del dataset crudo y
  deben ser generadas durante la etapa de transformación.

Este resultado confirma la compatibilidad estructural entre los datasets y
valida el uso de un proceso ETL específico para el año 2020, orientado a la
creación de variables derivadas y a la alineación final del esquema, garantizando
consistencia y comparabilidad temporal con el período pre-pandemia.


In [12]:
# Normalizar strings (por si hay espacios o vacíos)
df_2020["desde"] = df_2020["desde"].astype("string").str.strip().replace("", pd.NA)
df_2020["hasta"] = df_2020["hasta"].astype("string").str.strip().replace("", pd.NA)

# Derivar hora (soporta <NA>)
df_2020["hora_desde"] = pd.to_numeric(
    df_2020["desde"].str.split(":").str[0],
    errors="coerce"
).astype("Int8")

df_2020["hora_hasta"] = pd.to_numeric(
    df_2020["hasta"].str.split(":").str[0],
    errors="coerce"
).astype("Int8")

In [13]:
df_2020["hora_desde"].isna().sum(), df_2020["hora_hasta"].isna().sum()


(np.int64(1836), np.int64(1836))

In [14]:
df_2020["hora_desde"] = df_2020["hora_desde"].fillna(-1).astype("int64")
df_2020["hora_hasta"] = df_2020["hora_hasta"].fillna(-1).astype("int64")


In [16]:
df_2020["fecha"].isna().sum()


np.int64(1725156)

In [17]:
# Derivadas de fecha (soportan NaT)
df_2020["dia_semana"] = df_2020["fecha"].dt.day_name()

df_2020["mes"] = df_2020["fecha"].dt.month.astype("Int8")
df_2020["dia_mes"] = df_2020["fecha"].dt.day.astype("Int8")

df_2020["es_fin_semana"] = (
    df_2020["fecha"].dt.weekday.isin([5, 6]).astype("Int8")
)

df_2020["periodo"] = (
    df_2020["fecha"].dt.year * 100 + df_2020["fecha"].dt.month
).astype("Int32")


### IMPORTANTE

Durante la auditoría del dataset 2020 se detectó un volumen significativo
de registros sin fecha válida (~30% del total). Por este motivo, las
variables temporales derivadas (`mes`, `dia_mes`, `periodo`, etc.) se
implementaron utilizando tipos numéricos con soporte para valores nulos,
evitando la eliminación de registros y preservando la integridad del
dataset para análisis exploratorios y comparativos.
